# Ocean model data in Python: Xarray, Dask, Zarr and HEALPix

**Overview.**

Before geolocating a fish, we need to be comfortable with the *reference* data that `pangeo-fish` compares the tag with: a 3D ocean model.
In this notebook we open the **GLORYS** global ocean reanalysis (Global Ocean Physics Reanalysis, 1/12°, daily means) directly from the [Copernicus Marine Service](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030), look at it, plot it, compute with it and save it, using the core tools of the [Pangeo](https://pangeo.io) ecosystem:

- **Xarray**: labelled N-dimensional arrays (think of a `data.frame`, but with several dimensions such as time, depth, latitude and longitude).
- **Dask**: run the computation in parallel and only when needed (*lazy* evaluation), so that datasets larger than the memory can be handled.
- **Zarr**: a cloud-friendly, chunked file format for N-dimensional arrays. Copernicus Marine serves its data as Zarr, and `pangeo-fish` saves its intermediate results as Zarr.
- **HEALPix** (a Discrete Global Grid System, DGGS): an equal-area grid of the sphere. `pangeo-fish` does all its computations on this grid; we will convert GLORYS to HEALPix and plot it with `xdggs`.

**Time**: about 25 minutes. Run the cells one by one with `Shift+Enter`. Do not worry about understanding every argument: the goal is to recognise the objects you will meet again in the pangeo-fish notebook.

**Kernel**: use the kernel **Python (pangeo-fish)** (top right of the notebook).

## 0. Coming from R? A small dictionary

Most of you use R. Here is a rough correspondence between what you know and what you will see today.

| In R | In Python (today) | Comment |
|---|---|---|
| `library(dplyr)` | `import xarray as xr` | `xr` is the usual short name |
| `data.frame` / `tibble` | `pandas.DataFrame` | 2D table (not used much today) |
| `array` with named `dimnames`, `raster`/`terra` `SpatRaster`, `stars` object | `xarray.DataArray` / `xarray.Dataset` | N-dimensional labelled arrays. A `Dataset` is a dictionary of `DataArray`s sharing coordinates |
| `filter(df, lat > 46)` or `df[df$lat > 46, ]` | `ds.sel(latitude=slice(46, 59))` | select *by label* (coordinate value) |
| `df[1:10, ]` | `ds.isel(time=slice(0, 10))` | select *by integer position* |
| `mean(x, na.rm = TRUE)` | `ds.mean("time")` | reductions are along a *named* dimension; NaN are skipped by default |
| `ggplot2` | `matplotlib` (`.plot()`) or `hvplot` (interactive) | |
| `future` / `parallel` | `dask` | parallel and lazy computation |
| `ncdf4`, `terra::rast()` | `xr.open_dataset()`, `xr.open_zarr()` | file access |
| `x <- 1` | `x = 1` | assignment |
| `x$name` | `x["name"]` or `x.name` | attribute / item access |
| `TRUE`, `FALSE`, `NULL`, `NA` | `True`, `False`, `None`, `np.nan` | |

Two things that surprise R users:

1. **Indentation matters** in Python (it replaces `{ }`).
2. **Indexing starts at 0**, and `slice(0, 10)` means positions 0 to 9.

## 1. Setup

### Import the libraries

In [ ]:
import pathlib

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import hvplot.xarray  # adds the .hvplot accessor to xarray objects

import copernicusmarine
import xdggs
from healpix_resample import BilinearResampler

xr.set_options(display_expand_attrs=False)
print("xarray", xr.__version__, "| copernicusmarine", copernicusmarine.__version__, "| xdggs", xdggs.__version__)

### Start a Dask cluster

Dask splits the work into many small tasks and runs them in parallel on *workers*. On Pangeo@EOSC your server has several CPUs; we start a *local* cluster that uses them.

After running the cell, click on the **Dashboard** link (or open the Dask tab on the left side bar of JupyterLab and click on `Task Stream` and `Progress`): you will *see* the computations happen later in this notebook.

In [ ]:
from dask.distributed import Client

client = Client()  # a local cluster using the CPUs of this machine
client

### Log in to Copernicus Marine (once)

The Copernicus Marine Service is free but requires an account ([register here](https://data.marine.copernicus.eu/register)).
The cell below asks for your username and password **only the first time** and stores them in `~/.copernicusmarine/.copernicusmarine-credentials` in your home directory (which is persistent on Pangeo@EOSC). The next time you open a notebook, nothing is asked.

In [ ]:
credentials_file = pathlib.Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"

if credentials_file.exists():
    print("Copernicus Marine credentials found:", credentials_file)
else:
    copernicusmarine.login()  # prompts for username and password

## 2. Open GLORYS with Xarray

We open the daily-mean GLORYS reanalysis (`dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m"`, the same dataset that `pangeo-fish` uses by default as reference model) for:

- the **Celtic Sea / Bay of Biscay / Channel** area used in the pangeo-fish tutorial (`latitude` 46 to 59 °N, `longitude` -14 to 2 °E),
- **one week** of data,
- the variable **`thetao`** (sea water potential temperature) and **`zos`** (sea surface height).

Nothing is downloaded yet: `copernicusmarine.open_dataset` only reads the *metadata* and returns a lazy `xarray.Dataset`.

In [ ]:
%%time
bbox = {"latitude": [46, 59], "longitude": [-14, 2]}
time_slice = slice("2020-06-01", "2020-06-07")

glorys = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "zos"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
    start_datetime=time_slice.start,
    end_datetime=time_slice.stop,
)
glorys

Take a minute to explore the HTML representation above (click on the triangles and on the icons on the right):

- **Dimensions**: `time`, `depth`, `latitude`, `longitude` and their sizes.
- **Coordinates**: the labels along each dimension (dates, depths in metres, degrees).
- **Data variables**: `thetao` is 4D (time, depth, latitude, longitude), `zos` is 3D.
- **Attributes**: metadata (units, long names, source...).
- The data itself is a **Dask array**: look at the *chunk* size and the number of chunks. Each chunk is a small compressed piece of the Zarr store on the Copernicus cloud.

<div class="alert alert-success">
<b>Key points</b>
<ul>
<li>An <code>xarray.Dataset</code> holds several variables that share dimensions and coordinates.</li>
<li>Opening a dataset is instantaneous: the data is loaded only when needed.</li>
</ul>
</div>

### Select a piece of the data

With `.sel()` we select by *coordinate value* (label) and with `.isel()` by *position*. Let's take the sea surface temperature (first depth level, about 0.5 m) on the first day.

In [ ]:
sst = glorys["thetao"].isel(depth=0)  # DataArray: time, latitude, longitude
sst_day1 = sst.sel(time="2020-06-01")
sst_day1

### Plot

`xarray` comes with a `.plot()` method based on `matplotlib`: the dimension names and the units are used automatically. **Now** the data of that single map is downloaded from the Copernicus cloud (watch the Dask dashboard).

In [ ]:
sst_day1.plot(cmap="RdYlBu_r", figsize=(8, 6));

For an interactive map (zoom, hover to read the values, slider on time), use `hvplot`:

In [ ]:
sst.hvplot(
    x="longitude", y="latitude", cmap="RdYlBu_r", geo=True, coastline=True,
    frame_width=500, title="GLORYS sea surface temperature (degC)",
)

### A time series at one point

A tagged fish records temperature and depth. To compare these records with the model, `pangeo-fish` looks at the model values close to where the fish might be. The equivalent of that for *one* known position is a `.sel()` with `method="nearest"`:

In [ ]:
point = glorys["thetao"].sel(latitude=48.0, longitude=-5.0, method="nearest")
point.sel(depth=slice(0, 100)).plot(x="time", yincrease=False, cmap="RdYlBu_r", figsize=(8, 4))
plt.title("Temperature profile at 48N, 5W (depth 0 to 100 m)");

<div class="alert alert-success">
<b>Key points</b>
<ul>
<li><code>.sel()</code> selects by label, <code>.isel()</code> by position, <code>method="nearest"</code> picks the closest grid point.</li>
<li><code>.plot()</code> gives a static figure, <code>.hvplot()</code> an interactive one.</li>
</ul>
</div>

## 3. Dask: lazy and parallel computation

```{note}
Optional during the workshop: we will skip this section if we are short on time. Come back to it afterwards, it runs on its own.
```


Let's compute the mean surface temperature over the week. Writing the expression does **not** compute anything: xarray and Dask build a *task graph*.

In [ ]:
weekly_mean = sst.mean("time")
weekly_mean  # still lazy: look at the Dask array representation

`.compute()` executes the graph on the Dask cluster and returns the result in memory (as a NumPy array). Watch the dashboard while it runs: each small rectangle is a chunk being read and averaged by a worker.

In [ ]:
%%time
weekly_mean = weekly_mean.compute()
weekly_mean

In [ ]:
weekly_mean.plot(cmap="RdYlBu_r", figsize=(8, 6))
plt.title("Weekly mean sea surface temperature");

Everything else works the same way: `.std("time")`, `.max("depth")`, `(a - b)`, `.where(condition)`... You write the formula, Dask takes care of the chunks and the parallelism. `pangeo-fish` uses exactly this to compute the difference between the tag temperature and the model temperature for every grid point, every depth and every time step.

*Exercise (if time permits)*: compute the weekly mean of the full 3D field, `glorys["thetao"].mean("time")`, and watch the dashboard: many more chunks are involved.

<div class="alert alert-success">
<b>Key points</b>
<ul>
<li>Operations on Dask-backed arrays are lazy; <code>.compute()</code> (or plotting) triggers the work.</li>
<li>The Dask dashboard shows what is happening.</li>
</ul>
</div>

## 4. Zarr: saving (and re-opening) a dataset

```{note}
Optional during the workshop: we will skip this section if we are short on time. Come back to it afterwards, it runs on its own.
```


Zarr stores each chunk as a separate compressed file, with the metadata in small JSON files. It is the format used by Copernicus Marine in the cloud, and the format in which `pangeo-fish` writes all its intermediate results (`diff.zarr`, `emission.zarr`, ...), so that each step of the workflow can be resumed.

Let's save our one-week subset of the surface temperature locally. This downloads the data (still in parallel) and writes it to disk.

In [ ]:
%%time
# drop_encoding(): forget the compression settings of the remote store and let Zarr choose
sst.to_dataset(name="thetao").drop_encoding().to_zarr("glorys_sst.zarr", mode="w")

A Zarr store is a directory: let's see what is inside.

In [ ]:
store = pathlib.Path("glorys_sst.zarr")
files = sorted(p for p in store.rglob("*") if p.is_file())
print(len(files), "files, total", round(sum(p.stat().st_size for p in files) / 1e6, 1), "MB")
for p in files[:12]:
    print(p.relative_to(store))

And open it again with `xr.open_zarr()`. The dataset is lazy again (Dask-backed), exactly like the remote one: Zarr makes local and cloud data look the same.

In [ ]:
sst_local = xr.open_zarr("glorys_sst.zarr")
sst_local

<div class="alert alert-success">
<b>Key points</b>
<ul>
<li><code>.to_zarr()</code> writes, <code>xr.open_zarr()</code> reads; both are chunk-aware and parallel.</li>
<li>Remote (cloud) and local Zarr are handled the same way.</li>
</ul>
</div>

## 5. HEALPix: an equal-area grid for fish tracking

GLORYS is on a regular latitude/longitude grid. Such a grid has a problem for movement models: the cells shrink towards the poles (a 1/12° cell is about 9 km x 9 km at the equator, but 9 km x 4.5 km at 60°N). `pangeo-fish` limits how far a fish can swim per day; this is much easier when **all cells have the same area** and **neighbours are at the same distance**.

[HEALPix](https://healpix.sourceforge.io/) (Hierarchical Equal Area isoLatitude Pixelisation) divides the sphere into 12 base cells, each subdivided 4 times at each *refinement level*:

| level | number of cells | cell size (approx.) |
|---|---|---|
| 0 | 12 | 6 400 km |
| 5 | 12 288 | 200 km |
| 8 | 786 432 | 25 km |
| 10 | 12 582 912 | 6 km |
| 12 | 201 326 592 | 1.6 km |

HEALPix is one of the *Discrete Global Grid Systems* (DGGS); others are H3 (hexagons) or rHEALPix. The Python package [`xdggs`](https://github.com/xarray-contrib/xdggs) adds DGGS support to xarray: a dataset keeps a `cell_ids` coordinate instead of `latitude`/`longitude`, and `xdggs` knows how to compute the cell centres and polygons, neighbours, and how to plot them.

### Convert GLORYS to HEALPix

We use [`healpix-resample`](https://github.com/GRID4EARTH/healpix-resample), a small package developed in the [GRID4EARTH](https://grid4earth.eu/) project (and used by `pangeo-fish`), which interpolates values from any list of longitude/latitude points onto a HEALPix grid. Three steps:

1. give the resampler the longitudes and latitudes of the GLORYS grid points (it computes the interpolation weights once),
2. call `resample()` on the values, one row per time step,
3. put the result back into an `xarray.Dataset` with a `cell_ids` coordinate, and tell `xdggs` which grid it is.

The small function below does these three steps, so that the conversion is a one-line call. `BilinearResampler` is the simplest interpolation; the package also has nearest, bicubic, conservative and other resamplers with the same interface. `pangeo-fish` (`regrid_dataset_hpresample`, next notebook) combines several of them to deal properly with the coastline.

In [ ]:
def to_healpix(ds, level, var="thetao", ellipsoid="WGS84"):
    # interpolate a (time, latitude, longitude) variable onto a HEALPix grid
    # 1. the grid points, as 1D arrays of longitudes and latitudes
    lon2d, lat2d = np.meshgrid(ds["longitude"].values, ds["latitude"].values)
    resampler = BilinearResampler(
        lon_deg=lon2d.ravel().astype("float64"),
        lat_deg=lat2d.ravel().astype("float64"),
        level=level,
        verbose=False,
    )
    # 2. interpolate: rows = time steps, columns = grid points
    values = ds[var].values.reshape(ds.sizes["time"], -1)
    result = resampler.resample(values)
    # 3. back to xarray, with cell_ids instead of latitude/longitude
    grid_info = {"grid_name": "healpix", "level": level, "indexing_scheme": "nested", "ellipsoid": ellipsoid}
    return xr.Dataset(
        {var: (("time", "cell_ids"), result.cell_data, ds[var].attrs)},
        coords={"time": ds["time"], "cell_ids": ("cell_ids", result.cell_ids.astype("int64"), grid_info)},
    )


refinement_level = 8  # about 25 km cells; pangeo-fish typically uses 10 to 12
sst_healpix = to_healpix(sst_local.compute(), refinement_level)
sst_healpix

The dataset now has a single spatial dimension, `cell_ids`, instead of `latitude` and `longitude`. The `cell_ids` attributes describe the grid; `xdggs.decode()` reads them and gives us the `.dggs` accessor.

In [ ]:
sst_healpix = xdggs.decode(sst_healpix)
sst_healpix.dggs.grid_info

In [ ]:
# cell centres, computed by xdggs from the cell ids
sst_healpix.dggs.cell_centers()

### Plot on the HEALPix grid

`.dggs.explore()` draws each cell as a polygon on an interactive map (zoom in to see the equal-area cells; note that they are not aligned with the meridians).

In [ ]:
sst_healpix["thetao"].isel(time=0).dggs.explore(cmap="RdYlBu_r", alpha=0.8)

Save the HEALPix dataset as Zarr: this is exactly the kind of file (`diff-regridded.zarr`) that `pangeo-fish` produces in step 3 of the next notebook. `.dggs.encode("xdggs")` writes the grid description back into the `cell_ids` attributes, so that the file can be decoded again after reading.

In [ ]:
sst_healpix.dggs.encode("xdggs").drop_encoding().to_zarr("glorys_sst_healpix.zarr", mode="w")
xr.open_zarr("glorys_sst_healpix.zarr").pipe(xdggs.decode)

<div class="alert alert-success">
<b>Key points</b>
<ul>
<li>HEALPix cells all have the same area, which is what a movement model needs.</li>
<li><code>healpix-resample</code> (GRID4EARTH) interpolates any longitude/latitude data onto HEALPix in three lines.</li>
<li>In <code>xdggs</code>, a HEALPix dataset has a <code>cell_ids</code> coordinate with attributes describing the grid; <code>xdggs.decode()</code> activates the <code>.dggs</code> accessor (cell centres, polygons, <code>explore()</code>).</li>
<li><code>pangeo-fish</code> regrids the model-tag comparison to HEALPix, saves it as Zarr, and runs the geolocation on that grid.</li>
</ul>
</div>

## 6. Clean up

Close the Dask cluster before moving to the next notebook (each notebook starts its own).

In [ ]:
client.close()

## Going further

- [Xarray tutorial](https://tutorial.xarray.dev/) and the [Pangeo gallery](https://gallery.pangeo.io/).
- [Dask tutorial](https://tutorial.dask.org/).
- [Copernicus Marine toolbox documentation](https://toolbox-docs.marine.copernicus.eu/).
- [xdggs](https://xdggs.readthedocs.io/) and the [DGGS community](https://github.com/xarray-contrib/xdggs).
- [GRID4EARTH](https://grid4earth.eu/) packages: [healpix-geo](https://github.com/GRID4EARTH/healpix-geo) (cell ids, geometry), [healpix-resample](https://github.com/GRID4EARTH/healpix-resample) (used above), [healpix-convert](https://github.com/GRID4EARTH/healpix-convert) (batch conversion of whole Zarr datasets such as ERA5 or CAMS into the HEALPix DGGS Zarr format, with Dask), [healpix-plot](https://github.com/GRID4EARTH/healpix-plot) (static maps).
- The HEALPix paper: Górski et al. (2005), *HEALPix: A Framework for High-Resolution Discretization and Fast Analysis of Data Distributed on the Sphere*, ApJ 622, 759, [doi:10.1086/427976](https://doi.org/10.1086/427976).